In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

# A simple Backtracking Constraint Solver

A `CSP` (short for <em><u>C</u>onstraint <u>S</u>atisfaction <u>P</u>roblem</em>) is defined as a triple of the form
$$ \langle \texttt{Vars}, \texttt{Values}, \texttt{Constraints} \rangle $$
where
 * $\texttt{Vars}$ is a list of *variables*,
 * $\texttt{Values}$ is a list of *values* that these variables can take, and
 * $\texttt{Constraints}$ is a list of formulas describing *conditions*
   that the variables must satisfy.

   These formulas are given as strings.  These strings have to be valid
   <samp>OCaml</samp> expressions of type `bool`.  For example, the constraint $x \not= y$ is written as `"x <> y"` and the constraint $|x - y| = 2$ is written as `"abs (x - y) = 2"`.  Therefore, the variables of a CSP have to be valid <samp>OCaml</samp> identifiers, i.e. they have to start with a lower case letter.

To simplify the implementation we assume that the values are integers.  A CSP is represented as a record with three fields.  The order of the variables in the list `variables` determines the order in which values are assigned to the variables.

In [ ]:
type csp = {
  variables   : string list;
  values      : int list;
  constraints : string list;
}

An `assignment` maps **some** variables to values.  It is represented as a map of the module `RMap`, which is loaded by `setup.ml`.

In [ ]:
type assignment = (string, int) RMap.t

In TypeScript, the constraints are evaluated with the help of the constructor `Function`, which turns a string into a function.  <samp>OCaml</samp> is a compiled language.  However, the <samp>OCaml</samp> *toplevel*, which executes the cells of our notebooks, contains the complete <samp>OCaml</samp> compiler.  We can use it to compile a constraint into an <samp>OCaml</samp> function of type `assignment -> bool`.  This function is only called if all variables of the constraint have a value.

An `annotated_constraint` is a record consisting of the following fields:
- `formula` is the constraint as a string,
- `vars` is the set of all variables occurring in `formula`,
- `holds` is the compiled constraint, i.e. a function that takes an assignment and returns `true` if the constraint is satisfied.

In [ ]:
type annotated_constraint = {
  formula : string;
  vars    : string RSet.t;
  holds   : assignment -> bool;
}

The function `collect_variables` takes two arguments:
- `expr` is a string containing an <samp>OCaml</samp> expression,
- `variables` is the list of the variables of a CSP.

It returns the set of those variables that occur in `expr`.

**Implementation:** The TypeScript version collects all identifiers occurring in `expr` and removes the names of predefined functions.  As the variables of the CSP are known, we can do better:  The recursive function `identifiers i` returns the list of all identifiers that occur in `expr` from position `i` onwards.  An identifier starts with a letter or an underscore that is followed by letters, digits, underscores, or apostrophes.  `span j` returns the first position at or after `j` that does not contain such a character.  Numbers are skipped as a whole, so that the `e5` in `1e5` is not mistaken for an identifier.  Finally, only those identifiers are kept that are variables of the CSP.

In [ ]:
let collect_variables (expr : string) (variables : string list) : string RSet.t =
  let n = String.length expr in
  let is_start c = c = '_' || ('a' <= c && c <= 'z') || ('A' <= c && c <= 'Z') in
  let is_digit c = '0' <= c && c <= '9' in
  let is_char  c = is_start c || is_digit c || c = '\'' in
  let rec span j = if j < n && is_char expr.[j] then span (j + 1) else j in
  let rec identifiers i =
    if i >= n then []
    else if is_start expr.[i] then
      let j = span (i + 1) in
      String.sub expr i (j - i) :: identifiers j
    else if is_digit expr.[i] then identifiers (span (i + 1))
    else identifiers (i + 1)
  in
  RSet.of_list (List.filter (fun x -> List.mem x variables) (identifiers 0))

In [ ]:
collect_variables "abs (x - y) + abs (z1 - z2) = 0" ["x"; "y"; "z1"; "z2"; "w"]

The function `compile` takes two arguments:
- `formula` is a constraint, i.e. a string containing an <samp>OCaml</samp> expression of type `bool`,
- `vars` is the set of the variables occurring in `formula`.

It returns the function that checks whether an assignment satisfies `formula`.

**Implementation:** For the constraint `"x + y = z"` the function creates the following <samp>OCaml</samp> code:
```
compiled_constraint :=
  (fun (a : (string, int) RMap.t) ->
     let x = RMap.find "x" a in let y = RMap.find "y" a in let z = RMap.find "z" a in (x + y = z));;
```
This code defines a function that looks up the values of the variables in the assignment `a` and then evaluates the constraint.  The function is stored in the reference `compiled_constraint`.  The function `Toploop.use_silently` compiles and executes this code in the toplevel without printing anything.  If the code contains an error, e.g. if the constraint is not an expression of type `bool`, the toplevel prints an error message, `Toploop.use_silently` returns `false`, and `compile` raises an exception.  Otherwise, the new function is read from the reference `compiled_constraint`.

As the reference has the type `((string, int) RMap.t -> bool) ref`, the <samp>OCaml</samp> compiler checks that every compiled constraint has the right type.  The format `%S` of `Printf.sprintf` inserts a string literal, i.e. it adds the quotes.

In [ ]:
let compiled_constraint : (assignment -> bool) ref = ref (fun _ -> true)

let compile (formula : string) (vars : string RSet.t) : assignment -> bool =
  let bindings =
    String.concat ""
      (List.map (fun x -> Printf.sprintf "let %s = RMap.find %S a in " x x) (RSet.to_list vars))
  in
  let code =
    Printf.sprintf "compiled_constraint := (fun (a : (string, int) RMap.t) -> %s(%s));;" bindings formula
  in
  if Toploop.use_silently Format.err_formatter (Toploop.String code) then
    !compiled_constraint
  else
    failwith ("compile: the constraint " ^ formula ^ " is not an OCaml expression of type bool")

In [ ]:
let holds = compile "x + y = z" (RSet.of_list ["x"; "y"; "z"]);;
holds (RMap.of_list [("x", 3); ("y", 4); ("z", 7)])

The function `is_consistent` takes five arguments:
- `variable` is a variable,
- `value` is a value that is to be assigned to the variable `variable`,
- `assignment` is a partial variable assignment that does not assign a value for `variable`
  and that is *consistent* with all constraints in the list `constraints`,
- `assigned_vars` is the set of the variables that have a value in `assignment`,
- `constraints` is a list of annotated constraints.

The function checks whether the assignment
$$ \texttt{assignment} \cup \{\texttt{variable} \mapsto \texttt{value}\}$$
violates any of the constraints in `constraints`. It assumes that
`assignment` is *consistent*, i.e. all constraints that can already be evaluated with `assignment`
are evaluated as `true`.  Therefore, only those constraints have to be checked that contain the variable `variable` and whose variables all have a value.

In [ ]:
let is_consistent (variable : string) (value : int) (assignment : assignment)
                  (assigned_vars : string RSet.t) (constraints : annotated_constraint list) : bool =
  let new_assignment    = RMap.add variable value assignment in
  let new_assigned_vars = RSet.add variable assigned_vars in
  List.for_all
    (fun c ->
       let can_evaluate = RSet.mem variable c.vars && RSet.subset c.vars new_assigned_vars in
       not can_evaluate || c.holds new_assignment)
    constraints

The function `backtrack_search` takes five arguments:
- `assignment` is a **consistent** *partial variable assignment*,
- `assigned_vars` is the set of the variables that have a value in `assignment`,
- `variables` is the list of all variables,
- `values` is the list of all values,
- `constraints` is the list of annotated constraints.

It tries to extend the given assignment recursively and thereby produce a solution of the given CSP.  The result is `Some solution` if a solution is found, and `None` otherwise.

**Implementation:** If every variable has a value, the assignment is a solution.  Otherwise, the first variable `next_var` without a value is chosen, and the values are tried one after the other.  `List.find_map f l` applies `f` to the elements of `l` until `f` returns a result of the form `Some x`, which is then returned.  If `f` returns `None` for every element, `List.find_map` returns `None`.  Hence, as soon as a recursive call finds a solution, the remaining values are not tried any more.

In [ ]:
let rec backtrack_search (assignment : assignment) (assigned_vars : string RSet.t)
                         (variables : string list) (values : int list)
                         (constraints : annotated_constraint list) : assignment option =
  if RMap.cardinal assignment = List.length variables then
    Some assignment
  else
    let next_var = List.find (fun v -> not (RSet.mem v assigned_vars)) variables in
    List.find_map
      (fun value ->
         if is_consistent next_var value assignment assigned_vars constraints then
           backtrack_search (RMap.add next_var value assignment)
                            (RSet.add next_var assigned_vars)
                            variables values constraints
         else
           None)
      values

The input to the function `solve` is a *constraint satisfaction problem* `csp`.
The function `solve` tries to compute a solution of this problem via *backtracking*.
Its main purpose is to transform the given *CSP* into an *annotated CSP* where all the formulas
are *annotated* with their variables and compiled.  The function then calls the auxiliary function `backtrack_search` that recursively solves the *annotated CSP*.

In [ ]:
let solve (csp : csp) : assignment option =
  let annotate f =
    let vars = collect_variables f csp.variables in
    { formula = f; vars = vars; holds = compile f vars }
  in
  let annotated_constraints = List.map annotate csp.constraints in
  backtrack_search RMap.empty RSet.empty csp.variables csp.values annotated_constraints

In [ ]:
(* Example Usage to test the new function: *)
let my_csp = {
  variables   = ["x"; "y"; "z"];   (* Variables *)
  values      = [1; 2; 3; 4; 5];   (* Value Domain *)
  constraints = [                  (* Constraints *)
    "x * y = z + 1";
    "x + y = z";
    "x * y = 2 * z - x * x"
  ]
}

In [ ]:
match solve my_csp with
| Some solution ->
    print_endline "Solution found!";
    List.iter (fun v -> Printf.printf "%s = %d\n" v (RMap.find v solution)) ["x"; "y"; "z"]
| None ->
    print_endline "No solution exists."

A constraint that is not an <samp>OCaml</samp> expression of type `bool` is rejected when the CSP is solved.  The following cell fails on purpose.

In [ ]:
solve { variables = ["x"]; values = [1; 2]; constraints = ["x + 1"] }